# Reranker - Turkish Legal QA (A100)

**Proje:** CENG493 Turkish Legal RAG  
**Adim:** 3/5 - Reranker  
**GPU:** A100 (40GB)  
**Onceki:** Embedding Tuning (Hybrid R@5=81.3%, MRR=0.722)

**Proje gereksinimleri:**
- Cross-encoder fine-tuning
- Ranking optimization

**Pipeline:**  
Query -> Hybrid Retrieve (top-20) -> Rerank (cross-encoder) -> Top-5 -> LLM -> Answer

**Model:** BAAI/bge-reranker-v2-m3

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate bitsandbytes rank_bm25 snowballstemmer rouge-score nltk

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, re, math
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/493_project")

import sys
sys.path.insert(0, str(PROJECT_DIR))
from evaluation.metrics_utils import (
    RAG_SYSTEM_PROMPT,
    format_context_blocks_for_llm,
    aggregate_qa_row,
    retrieval_hit,
)

chunks = []
with open(PROJECT_DIR / "data/processed/chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))
chunk_texts = [c["text"] for c in chunks]

with open(PROJECT_DIR / "evaluation/gold_test_set.json", "r", encoding="utf-8") as f:
    gold_questions = json.load(f)
answerable_qs = [q for q in gold_questions if q.get("answerable_from_corpus", True)]

with open(PROJECT_DIR / "evaluation/baseline_results.json", "r", encoding="utf-8") as f:
    baseline_results = json.load(f)

with open(PROJECT_DIR / "evaluation/embedding_tuning_results.json", "r", encoding="utf-8") as f:
    emb_results = json.load(f)

with open(PROJECT_DIR / "data/processed/embedding_train_triplets.json", "r", encoding="utf-8") as f:
    train_triplets = json.load(f)

print(f"Chunks: {len(chunks)}")
print(f"Answerable Qs: {len(answerable_qs)}")
print(f"Training triplets: {len(train_triplets)}")
print(f"Baseline R@5: {baseline_results['metrics']['recall@5']:.4f}")
print(f"Emb Tuning Hybrid R@5: {emb_results['metrics_tuned_hybrid']['recall@5']:.4f}")

In [ ]:
import faiss
import snowballstemmer
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

tuned_model_path = str(PROJECT_DIR / "models" / "bge-m3-legal-v2")
embed_model = SentenceTransformer(tuned_model_path, device="cuda")
print(f"Tuned embedding model loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

tuned_index = faiss.read_index(str(PROJECT_DIR / "models" / "embedding_tuned_v2" / "faiss.index"))
tuned_vecs = np.load(str(PROJECT_DIR / "models" / "embedding_tuned_v2" / "dense_vecs.npy"))
print(f"FAISS index: {tuned_index.ntotal} vectors")

tr_stemmer = snowballstemmer.stemmer('turkish')
TURKISH_STOPWORDS = {
    "bir", "bu", "de", "da", "ve", "ile", "için", "olan", "olarak",
    "veya", "ya", "hem", "ne", "her", "o", "şu", "ki", "mi", "mu",
    "mı", "mü", "ise", "den", "dan", "ten", "tan", "dir", "dır",
    "dür", "dur", "tir", "tır", "tur", "tür", "gibi", "kadar",
    "sonra", "önce", "üzere", "göre", "ait", "dair", "en", "çok",
    "var", "yok", "olan", "olup", "ancak", "fakat", "ama",
}

def tokenize_tr(text: str) -> list:
    tokens = re.findall(r'\w+', text.lower())
    tokens = [t for t in tokens if t not in TURKISH_STOPWORDS and len(t) > 1]
    return tr_stemmer.stemWords(tokens)

tokenized_chunks = [tokenize_tr(t) for t in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)
print(f"BM25 index ready: {len(tokenized_chunks)} documents")


def _make_result(idx: int, rank: int, score: float) -> dict:
    c = chunks[idx]
    return {
        "rank": rank, "score": score,
        "chunk_id": c["chunk_id"], "source": c["source"],
        "article_title": c.get("article_title", ""),
        "article_key": c.get("article_key", ""),
        "text": c["text"],
    }


def retrieve_hybrid(query: str, top_k: int = 20, alpha: float = 0.8) -> list:
    q_vec = embed_model.encode([query], normalize_embeddings=True)
    q_vec = np.array(q_vec).astype("float32")

    dense_scores = np.dot(tuned_vecs, q_vec.T).flatten()
    bm25_scores = bm25.get_scores(tokenize_tr(query))

    d_min, d_max = dense_scores.min(), dense_scores.max()
    dense_norm = (dense_scores - d_min) / (d_max - d_min + 1e-9)

    b_max = bm25_scores.max()
    bm25_norm = bm25_scores / (b_max + 1e-9)

    combined = alpha * dense_norm + (1 - alpha) * bm25_norm
    top_indices = np.argsort(combined)[::-1][:top_k]

    return [_make_result(int(idx), r+1, float(combined[idx]))
            for r, idx in enumerate(top_indices)]


test_r = retrieve_hybrid("Egemenlik kime aittir?", top_k=5)
for r in test_r:
    print(f"  R{r['rank']} ({r['score']:.4f}) [{r['source']} | {r['article_title']}]")

## 1. Base Reranker

**BAAI/bge-reranker-v2-m3** cross-encoder'i once fine-tune etmeden uyguluyoruz.  
Pipeline: Hybrid top-20 -> Rerank -> top-5

In [ ]:
from sentence_transformers import CrossEncoder

base_reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda")
print(f"Base reranker loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")


def rerank(query: str, retrieved: list, reranker, top_k: int = 5) -> list:
    if not retrieved:
        return []
    pairs = [(query, r["text"]) for r in retrieved]
    scores = reranker.predict(pairs, show_progress_bar=False)
    ranked = sorted(zip(scores, retrieved), key=lambda x: -x[0])
    results = []
    for rank, (score, r) in enumerate(ranked[:top_k]):
        results.append({**r, "rank": rank + 1, "rerank_score": float(score)})
    return results


print("\nTest: Reranking 'Egemenlik kime aittir?'")
candidates = retrieve_hybrid("Egemenlik kime aittir?", top_k=20)
reranked = rerank("Egemenlik kime aittir?", candidates, base_reranker, top_k=5)
for r in reranked:
    print(f"  R{r['rank']} (rerank={r['rerank_score']:.4f}) [{r['source']} | {r['article_title']}]")

## 2. Cross-Encoder Fine-tuning

Embedding tuning asamasinda olusturulan (query, positive, negative) triplet'lerini kullanarak  
cross-encoder'i hukuk alanina adapte ediyoruz.

- Positive ciftler: label=1.0
- Negative ciftler: label=0.0
- epochs=2, batch_size=16, lr=2e-5

In [ ]:
import random
from sentence_transformers import InputExample
from torch.utils.data import DataLoader

random.seed(42)
random.shuffle(train_triplets)
MAX_TRAIN = min(len(train_triplets), 2000)
selected = train_triplets[:MAX_TRAIN]

train_samples = []
for t in selected:
    train_samples.append(InputExample(texts=[t["query"], t["positive"]], label=1.0))
    train_samples.append(InputExample(texts=[t["query"], t["negative"]], label=0.0))

random.shuffle(train_samples)
print(f"Training samples: {len(train_samples)} (from {len(selected)} triplets)")

BATCH_SIZE = 16
EPOCHS = 2
LR = 2e-5

train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=BATCH_SIZE)
WARMUP = int(len(train_dataloader) * EPOCHS * 0.1)

save_reranker_path = str(PROJECT_DIR / "models" / "reranker-legal")

print(f"batch_size={BATCH_SIZE}, epochs={EPOCHS}, lr={LR}, warmup={WARMUP}")
print(f"Batches/epoch: {len(train_dataloader)}")

tuned_reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda")

tuned_reranker.fit(
    train_dataloader=train_dataloader,
    epochs=EPOCHS,
    warmup_steps=WARMUP,
    optimizer_params={"lr": LR},
    output_path=save_reranker_path,
    show_progress_bar=True,
)

print(f"\nFine-tuning complete! Saved -> {save_reranker_path}")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 3. Evaluation

Karsilastirilacak konfigurasyonlar:
1. Baseline (onceki sonuclardan)
2. Embedding Tuning Hybrid (onceki sonuclardan)
3. **+ Base Reranker** (yeni)
4. **+ Fine-tuned Reranker** (yeni)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
print(f"LLM loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")


def rag_answer(question: str, retrieved: list) -> str:
    context = format_context_blocks_for_llm(retrieved, max_chunks=5)
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": f"Baglam:\n{context}\n\nSoru: {question}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=4096).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=384, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import nltk

for _p in ("punkt", "punkt_tab"):
    try:
        nltk.download(_p, quiet=True)
    except Exception:
        pass

print("Eval: evaluation.metrics_utils")


In [ ]:
from tqdm import tqdm

RETRIEVE_K = 20
RERANK_K = 5

results_base_rr = []
results_tuned_rr = []

for q in tqdm(answerable_qs, desc="Evaluating"):
    candidates = retrieve_hybrid(q["soru"], top_k=RETRIEVE_K)

    rr_base = rerank(q["soru"], candidates, base_reranker, top_k=RERANK_K)
    rr_tuned = rerank(q["soru"], candidates, tuned_reranker, top_k=RERANK_K)

    rh_base = retrieval_hit(rr_base, q)
    rh_tuned = retrieval_hit(rr_tuned, q)

    ans_base = rag_answer(q["soru"], rr_base)
    ans_tuned = rag_answer(q["soru"], rr_tuned)
    qa_b = aggregate_qa_row(ans_base, q["cevap"], rr_base, q)
    qa_t = aggregate_qa_row(ans_tuned, q["cevap"], rr_tuned, q)
    pb_b = qa_b.pop("pred_body")
    pb_t = qa_t.pop("pred_body")

    base_info = {
        "id": q["id"], "soru": q["soru"],
        "gold_cevap": q["cevap"], "kaynak": q["kaynak"],
        "difficulty": q["difficulty"], "type": q["type"],
    }

    results_base_rr.append({**base_info, **rh_base, "pred_cevap": ans_base, "pred_body": pb_b, **qa_b})
    results_tuned_rr.append({**base_info, **rh_tuned, "pred_cevap": ans_tuned, "pred_body": pb_t, **qa_t})

print(f"\nDone: {len(results_tuned_rr)} questions evaluated")

In [ ]:
df_base_rr = pd.DataFrame(results_base_rr)
df_tuned_rr = pd.DataFrame(results_tuned_rr)
bl = baseline_results["metrics"]
et = emb_results["metrics_tuned_hybrid"]

print("=" * 75)
print("  RERANKER SONUCLARI")
print("=" * 75)

print(f"\n{'Metric':<12} {'Baseline':>10} {'Emb Hybrid':>12} {'+ Base RR':>12} {'+ Tuned RR':>12}")
print("-" * 60)
for k in [1, 3, 5, 10]:
    b = bl[f"recall@{k}"]
    e = et[f"recall@{k}"]
    br = df_base_rr[f"hit@{k}"].mean()
    tr = df_tuned_rr[f"hit@{k}"].mean()
    print(f"Recall@{k:<2d}    {b:>9.4f}  {e:>11.4f}  {br:>11.4f}  {tr:>11.4f}")

b_mrr = bl["mrr"]
e_mrr = et["mrr"]
br_mrr = df_base_rr["mrr"].mean()
tr_mrr = df_tuned_rr["mrr"].mean()
print(f"{'MRR':<12} {b_mrr:>9.4f}  {e_mrr:>11.4f}  {br_mrr:>11.4f}  {tr_mrr:>11.4f}")

bl_ndcg = bl.get("ndcg@10", float("nan"))
et_ndcg = et.get("ndcg@10", float("nan"))
br_ndcg = df_base_rr["ndcg@10"].mean()
tr_ndcg = df_tuned_rr["ndcg@10"].mean()
print(f"{'nDCG@10':<12} {bl_ndcg:>9.4f}  {et_ndcg:>11.4f}  {br_ndcg:>11.4f}  {tr_ndcg:>11.4f}")

best_rr = "Tuned" if tr_mrr >= br_mrr else "Base"
best_df = df_tuned_rr if best_rr == "Tuned" else df_base_rr
best_f1 = best_df["f1"].mean()
best_em = best_df["em"].mean()

print(f"\n{'Metric':<12} {'Baseline':>10} {'Emb Hybrid':>12} {'Best RR ('+best_rr+')':>14} {'Delta':>8}")
print("-" * 58)
print(f"{'F1':<12} {bl['f1']:>9.4f}  {et['f1']:>11.4f}  {best_f1:>13.4f}  {best_f1-bl['f1']:>+7.4f}")
print(f"{'EM':<12} {bl['em']:>9.4f}  {et['em']:>11.4f}  {best_em:>13.4f}  {best_em-bl['em']:>+7.4f}")
for col, lab in [("bleu", "BLEU"), ("rouge1", "ROUGE-1"), ("rouge2", "ROUGE-2"), ("rougeL", "ROUGE-L"),
                 ("faithfulness_token_recall", "Faithful."), ("citation_precision_retrieved", "CitePrec")]:
    bv, ev, bv2 = bl.get(col, float("nan")), et.get(col, float("nan")), best_df[col].mean()
    print(f"{lab:<12} {bv:>9.4f}  {ev:>11.4f}  {bv2:>13.4f}")
_bgr = best_df.loc[best_df["citation_recall_gold"] >= 0, "citation_recall_gold"]
print(f"{'CiteRecGold':<12} {'(n/a)':>9}  {'(n/a)':>11}  {_bgr.mean():>13.4f}  (n={len(_bgr)})")

print(f"\n--- Difficulty (Best Reranker: {best_rr}) ---")
for d in ["easy", "medium", "hard", "very hard"]:
    s = best_df[best_df["difficulty"] == d]
    if len(s):
        print(f"  {d:10s}: R@5={s['hit@5'].mean():.3f}  F1={s['f1'].mean():.3f}  nDCG={s['ndcg@10'].mean():.3f}  (n={len(s)})")

print(f"\n--- Question Type (Best Reranker: {best_rr}) ---")
for t in sorted(best_df["type"].unique()):
    s = best_df[best_df["type"] == t]
    print(f"  {t:15s}: R@5={s['hit@5'].mean():.3f}  F1={s['f1'].mean():.3f}  nDCG={s['ndcg@10'].mean():.3f}  (n={len(s)})")

In [ ]:
output = {
    "experiment": "reranker",
    "embedding": "BAAI/bge-m3 (legal fine-tuned)",
    "retrieval": "hybrid_dense_bm25 (alpha=0.8)",
    "reranker_base": "BAAI/bge-reranker-v2-m3",
    "reranker_tuned": "BAAI/bge-reranker-v2-m3 (legal fine-tuned)",
    "llm": "Qwen/Qwen2.5-7B-Instruct-4bit",
    "pipeline": "hybrid_top20 -> rerank -> top5 -> LLM",
    "training": {
        "samples": len(train_samples),
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
    },
    "num_questions": len(results_tuned_rr),
    "best_reranker": best_rr,
    "metrics_base_reranker": {
        f"recall@{k}": float(df_base_rr[f"hit@{k}"].mean()) for k in [1,3,5,10]
    } | {
        "mrr": float(br_mrr),
        "ndcg@10": float(br_ndcg),
        "f1": float(df_base_rr["f1"].mean()),
        "em": float(df_base_rr["em"].mean()),
        "bleu": float(df_base_rr["bleu"].mean()),
        "rouge1": float(df_base_rr["rouge1"].mean()),
        "rouge2": float(df_base_rr["rouge2"].mean()),
        "rougeL": float(df_base_rr["rougeL"].mean()),
        "faithfulness_token_recall": float(df_base_rr["faithfulness_token_recall"].mean()),
        "citation_precision_retrieved": float(df_base_rr["citation_precision_retrieved"].mean()),
        "citation_recall_gold_mean": float(
            df_base_rr.loc[df_base_rr["citation_recall_gold"] >= 0, "citation_recall_gold"].mean()
        ) if (df_base_rr["citation_recall_gold"] >= 0).any() else None,
    },
    "metrics_tuned_reranker": {
        f"recall@{k}": float(df_tuned_rr[f"hit@{k}"].mean()) for k in [1,3,5,10]
    } | {
        "mrr": float(tr_mrr),
        "ndcg@10": float(tr_ndcg),
        "f1": float(df_tuned_rr["f1"].mean()),
        "em": float(df_tuned_rr["em"].mean()),
        "bleu": float(df_tuned_rr["bleu"].mean()),
        "rouge1": float(df_tuned_rr["rouge1"].mean()),
        "rouge2": float(df_tuned_rr["rouge2"].mean()),
        "rougeL": float(df_tuned_rr["rougeL"].mean()),
        "faithfulness_token_recall": float(df_tuned_rr["faithfulness_token_recall"].mean()),
        "citation_precision_retrieved": float(df_tuned_rr["citation_precision_retrieved"].mean()),
        "citation_recall_gold_mean": float(
            df_tuned_rr.loc[df_tuned_rr["citation_recall_gold"] >= 0, "citation_recall_gold"].mean()
        ) if (df_tuned_rr["citation_recall_gold"] >= 0).any() else None,
    },
    "baseline_metrics": bl,
    "emb_tuning_metrics": et,
    "per_question_base_rr": results_base_rr,
    "per_question_tuned_rr": results_tuned_rr,
}

save_path = PROJECT_DIR / "evaluation" / "reranker_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Results saved -> {save_path}")
print("\nSonraki adim: LLM Fine-tuning (Qwen2.5-7B + QLoRA)")